In [ ]:
import h5py
from temporaldata import Data
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import StratifiedKFold

In [ ]:
md = pd.read_csv("../neuron_metadata/allen_vc_2019.csv")
md.unit_id = md.unit_id.astype(str)

# Inductive splits

In [ ]:
ssl_ids = md.unit_id.values

labelled_md = md[md.lolcat_celltype != "wt"]
session_ids = np.unique(labelled_md.session_id)

# Create a session_id : cell_type mapping
session_ctypes = []
for sid in session_ids:
    ctypes = labelled_md[labelled_md.session_id == sid].lolcat_celltype.values
    assert len(np.unique(ctypes)) == 1  # confirm the assumption that only 1 kind of cell type is available in each session
    session_ctypes.append(ctypes[0])
session_ctype_map = pd.DataFrame(session_ctypes, session_ids, columns=["celltype"])

classifier_splits = []
for sid in session_ids:
    test_mask = (labelled_md.session_id == sid).to_numpy()
    test_ids = labelled_md.unit_id.to_numpy()[test_mask]
    
    trainval_mask = ~test_mask
    trainval_idx = np.where(trainval_mask)[0] 
    
    remaining_sids = session_ids[session_ids != sid]
    remaining_ctypes = session_ctype_map.loc[remaining_sids].to_numpy().ravel()
    
    # Inner K-fold
    cv_splits = []
    kfold = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
    for train_idx, val_idx in kfold.split(remaining_sids, remaining_ctypes):
        train_sids, val_sids = remaining_sids[train_idx], remaining_sids[val_idx]
        train_ids = labelled_md[np.isin(labelled_md.session_id, train_sids)].unit_id.to_numpy()
        val_ids = labelled_md[np.isin(labelled_md.session_id, val_sids)].unit_id.to_numpy()
        cv_splits.append({"train": train_ids, "val": val_ids})
    
    classifier_splits.append({"test": test_ids, "train_cv": cv_splits})

label_map = pd.DataFrame(labelled_md.lolcat_celltype.to_numpy(), labelled_md.unit_id.to_numpy(), columns=["label"])
data = {
    "splits": classifier_splits,
    "label_map": label_map,
}

output_file = "../splits/allen_vc_classifier.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")